# Annotator Performance Over Time

This notebook tracks inter-annotator agreement **across evaluation project cycles**,
showing how annotation quality evolved through the
*evaluation → agreement measurement → calibration meeting → next evaluation* loop.

It is the third notebook in the series:

1. `Translation_Annotator_Agreement.ipynb` — deep dive on a single project
2. `Translation_Quality_Comparison.ipynb` — compare translations of one document
3. **This notebook** — longitudinal view across all projects

**Metrics tracked per project:**
- Exact span match rate
- Span agreement (character-level F1)
- Cohen's κ (error category agreement on overlapping spans)
- Lead time (mean minutes per annotation)
- Annotation density (mean spans marked per annotation, per annotator)
- Document-level rating agreement (mean absolute difference on correspondence & readability)

All parsing and metric logic is copied from the single-project notebook so numbers
are directly comparable across the series. One deliberate difference: task IDs are
namespaced per file (`filename:task_id`) so that projects with two translations
(two Label Studio exports) never accidentally merge annotations from different
translations into one document.

## 1. Project Config

Edit this cell each time a new evaluation cycle completes.

- `PROJECTS` — list of evaluation rounds **in chronological order**. Each entry needs:
  - `label`: short display name used on the x-axis (e.g., `"TQE-5"`)
  - `date`: round date (`YYYY-MM-DD`) — shown under the label
  - `files`: list of one or more Label Studio JSON exports for that round.
    Early single-translation projects have one file; later projects with two
    translations have two. Metrics are pooled across all files in the round.
- `CALIBRATION_AFTER` — labels of projects that were followed by a calibration
  meeting. A dashed vertical marker is drawn after each on every chart, so you
  can see whether agreement moved after each meeting.
- `TARGETS` — horizontal reference lines drawn on the trend charts. Set a value
  to `None` to hide that line.

In [ ]:
# ── PROJECT CONFIG ────────────────────────────────────────────────────────────

PROJECT_NAME = "TQE Annotator Performance"
REPORT_DATE  = "2026-06-10"   # update each run
OUTPUT_DIR   = "./reports"
DATA_DIR     = "./data"

PROJECTS = [
    {
        "label": "Label 1",
        "date":  "2026-06-03",
        "files": [f"{DATA_DIR}/TQE-Project-1.json"],
    },
    {
        "label": "Label 2",
        "date":  "2026-06-10",
        "files": [f"{DATA_DIR}/TQE-Project-2.json"],
    },
    {
        "label": "Label 3",
        "date":  "2026-06-17",
        "files": [
            f"{DATA_DIR}/TQE-Project-3-1.json",
            f"{DATA_DIR}/TQE-Project-3-2.json",
        ],
    },
    # Add more projects as needed, following the same structure
]

# Projects followed by a calibration meeting (marker drawn after these points).
# A meeting followed every round; replace with an explicit list of labels
# (e.g., ["TQE-2", "TQE-5"]) if that ever changes.
CALIBRATION_AFTER = [p["label"] for p in PROJECTS]

# Horizontal target lines on trend charts (None = no line)
TARGETS = {
    "exact_match": 0.60,   # project goal: 60% exact span agreement
    "span_f1":     0.60,   # matches the ≥0.60 target in the comparison report
    "kappa":       0.60,   # category agreement floor
    "lead_time":   None,   # no target — tracked for context, not as a goal
}

## 2. Imports

In [ ]:
import json
import os
import datetime
import io
import base64
from itertools import combinations

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score
from IPython.display import display, HTML

sns.set_theme(style="whitegrid")
matplotlib.rcParams["figure.dpi"] = 100

## 3. Data Loading and Parsing

Same Label Studio parsing as the single-project notebook, with one change:
`parse_label_studio_export` keys documents by `"{filename}:{task_id}"` instead of
the bare task ID. Two exports of the *same* round (two translations) can reuse
Label Studio task IDs, and merging them would create false cross-translation
annotator pairs. Namespacing keeps every (translation, document) separate while
still pooling all of a round's pairs into its metrics.

In [ ]:
def parse_result_array(result):
    """
    Flatten a Label Studio annotation's result array into a structured dict.

    Span items share a region ID — we group them before building enriched_labels.
    Document-level fields are collected separately.
    Relation items (from_name=None) are silently skipped.
    """
    regions = {}  # region_id → accumulated field values

    doc_fields = {
        "overall_correspondence":  None,
        "overall_readability":     None,
        "document_issues":         None,
        "correspondence_comments": None,
        "readability_comments":    None,
    }

    for item in result:
        from_name = item.get("from_name")
        if from_name is None:
            continue  # relation items

        value     = item.get("value", {})
        region_id = item.get("id")

        # ── Span-level fields ──────────────────────────────────────────────
        if from_name == "label":
            if region_id not in regions:
                regions[region_id] = {
                    "start": value.get("start"),
                    "end":   value.get("end"),
                    "text":  value.get("text", ""),
                }
            regions[region_id]["labels"] = value.get("labels", [])

        elif from_name == "impact":
            if region_id not in regions:
                regions[region_id] = {}
            choices = value.get("choices", [])
            regions[region_id]["impact"] = choices[0] if choices else None

        elif from_name == "subcategories":
            if region_id not in regions:
                regions[region_id] = {}
            choices = value.get("choices", [])
            regions[region_id]["subcategory"] = choices[0] if choices else None

        elif from_name == "comments":
            if region_id not in regions:
                regions[region_id] = {}
            texts = value.get("text", [])
            regions[region_id]["comments"] = texts[0] if texts else None

        # ── Document-level fields ──────────────────────────────────────────
        elif from_name == "overall_correspondence":
            doc_fields["overall_correspondence"] = value.get("rating")

        elif from_name == "overall_readability":
            doc_fields["overall_readability"] = value.get("rating")

        elif from_name in ("document_issues", "correspondence_comments", "readability_comments"):
            texts = value.get("text", [])
            doc_fields[from_name] = texts[0] if texts else None

    # Build enriched_labels — only regions that have a label entry
    enriched_labels = []
    for region_id, region in regions.items():
        if "labels" not in region:
            continue
        enriched_labels.append({
            "start":       region.get("start"),
            "end":         region.get("end"),
            "text":        region.get("text", ""),
            "labels":      region.get("labels", []),
            "subcategory": region.get("subcategory"),
            "impact":      region.get("impact"),
            "comments":    region.get("comments"),
        })

    return {"enriched_labels": enriched_labels, **doc_fields}


def parse_label_studio_export(file_paths):
    """
    Load one or more Label Studio JSON exports and return:
        task_key_to_annotations  dict[str, list[dict]]

    Keys are namespaced as "{filename}:{task_id}" so multiple exports
    (e.g., two translations annotated in the same round) never merge.
    """
    task_key_to_annotations = {}

    for file_path in file_paths:
        if not os.path.isfile(file_path):
            raise FileNotFoundError(f"Annotation file not found: {file_path}")

        fname = os.path.basename(file_path)
        print(f"  Reading: {fname}")
        with open(file_path, "r", encoding="utf-8") as f:
            tasks = json.load(f)
        print(f"    {len(tasks)} task(s)")

        for task in tasks:
            task_key = f"{fname}:{task['id']}"
            text     = task["data"]["text"]

            if task_key not in task_key_to_annotations:
                task_key_to_annotations[task_key] = []

            for raw_ann in task.get("annotations", []):
                if raw_ann.get("was_cancelled"):
                    continue

                parsed = parse_result_array(raw_ann.get("result", []))

                review_time_seconds = None
                try:
                    created = datetime.datetime.fromisoformat(
                        raw_ann["created_at"].replace("Z", "+00:00"))
                    updated = datetime.datetime.fromisoformat(
                        raw_ann["updated_at"].replace("Z", "+00:00"))
                    review_time_seconds = (updated - created).total_seconds()
                except (KeyError, ValueError, TypeError):
                    pass

                annotation = {
                    "task_id":             task_key,
                    "annotation_id":       raw_ann["id"],
                    "annotator_id":        raw_ann["completed_by"],
                    "text":                text,
                    "lead_time":           raw_ann.get("lead_time"),
                    "lead_time_minutes":   round((raw_ann.get("lead_time") or 0) / 60, 2),
                    "created_at":          raw_ann.get("created_at"),
                    "updated_at":          raw_ann.get("updated_at"),
                    "review_time":         review_time_seconds,
                    **parsed,
                }
                task_key_to_annotations[task_key].append(annotation)

    return task_key_to_annotations

In [ ]:
# Load every project round
for proj in PROJECTS:
    print(f"\nLoading {proj['label']} ({proj['date']}) — {len(proj['files'])} file(s)")
    proj["data"] = parse_label_studio_export(proj["files"])
    n_annotations = sum(len(v) for v in proj["data"].values())
    annotators    = sorted({a["annotator_id"] for v in proj["data"].values() for a in v})
    proj["annotators"] = annotators
    print(f"  → {len(proj['data'])} documents, {n_annotations} annotations, "
          f"annotators: {annotators}")

## 4. Agreement Metrics

The three agreement calculations below are copied from the single-project notebook
(§5.1–5.3) so values match across the series:

- **Exact match** — identical (start, end) boundaries; rate = matches / unique spans
- **Span F1** — character-level precision/recall between annotator pairs
- **Cohen's κ** — category agreement on spans overlapping ≥ 50%, weighted by pair size

Plus longitudinal additions:

- **Lead time** — mean Label Studio `lead_time` per annotation, in minutes
- **Annotation density** — mean spans marked per annotation (overall and per annotator).
  Useful context: a sudden drop in density can inflate exact match while hiding
  under-annotation.
- **Rating agreement** — mean absolute difference between annotators' document-level
  correspondence and readability ratings (0 = perfect agreement on the 1–4 scale).

In [ ]:
def calculate_exact_matches(task_id_to_annotations):
    """
    Calculate exact matches between annotators for each document.
    Returns dict with per-document and overall exact match statistics.
    """
    results = {
        'per_document': {},
        'overall': {
            'total_comparisons': 0,
            'total_exact_matches': 0,
            'exact_match_rate': 0
        }
    }

    for doc_id, annotations in task_id_to_annotations.items():
        annotator_to_spans = {}
        for annotation in annotations:
            annotator = annotation.get('annotator_id')
            if not annotator:
                continue
            spans = []
            for label in annotation.get('enriched_labels', []):
                spans.append({'start': label.get('start'), 'end': label.get('end')})
            annotator_to_spans[annotator] = spans

        if len(annotator_to_spans) < 2:
            continue

        doc_results = {
            'annotator_pairs': [],
            'total_comparisons': 0,
            'total_exact_matches': 0,
            'exact_match_rate': 0
        }

        for annotator1, annotator2 in combinations(annotator_to_spans.keys(), 2):
            spans1 = annotator_to_spans[annotator1]
            spans2 = annotator_to_spans[annotator2]

            spans1_set = {(s['start'], s['end']) for s in spans1}
            spans2_set = {(s['start'], s['end']) for s in spans2}

            exact_matches = len(spans1_set & spans2_set)
            unique_spans  = len(spans1_set | spans2_set)
            rate = exact_matches / unique_spans if unique_spans > 0 else 0

            doc_results['annotator_pairs'].append({
                'annotator1': annotator1, 'annotator2': annotator2,
                'spans1': len(spans1), 'spans2': len(spans2),
                'unique_spans': unique_spans,
                'exact_matches': exact_matches,
                'exact_match_rate': rate,
            })
            doc_results['total_comparisons']   += unique_spans
            doc_results['total_exact_matches'] += exact_matches

        if doc_results['total_comparisons'] > 0:
            doc_results['exact_match_rate'] = (
                doc_results['total_exact_matches'] / doc_results['total_comparisons'])

        results['per_document'][doc_id] = doc_results
        results['overall']['total_comparisons']   += doc_results['total_comparisons']
        results['overall']['total_exact_matches'] += doc_results['total_exact_matches']

    if results['overall']['total_comparisons'] > 0:
        results['overall']['exact_match_rate'] = (
            results['overall']['total_exact_matches'] /
            results['overall']['total_comparisons'])

    return results

In [ ]:
def calculate_span_f1(task_id_to_annotations):
    """
    Character-level F1 between annotator pairs (same logic as the
    single-project notebook). Returns per-document and overall averages.
    """
    results = {
        'per_document': {},
        'overall': {'total_comparisons': 0, 'avg_precision': 0,
                    'avg_recall': 0, 'avg_f1': 0}
    }
    overall_p, overall_r, overall_f = [], [], []

    for doc_id, annotations in task_id_to_annotations.items():
        annotator_to_spans = {}
        for annotation in annotations:
            annotator = annotation.get('annotator_id')
            if not annotator:
                continue
            spans = [{'start': l.get('start'), 'end': l.get('end')}
                     for l in annotation.get('enriched_labels', [])]
            annotator_to_spans[annotator] = spans

        if len(annotator_to_spans) < 2:
            continue

        doc_results = {'annotator_pairs': [], 'avg_precision': 0,
                       'avg_recall': 0, 'avg_f1': 0}
        doc_p, doc_r, doc_f = [], [], []

        for annotator1, annotator2 in combinations(annotator_to_spans.keys(), 2):
            spans1 = annotator_to_spans[annotator1]
            spans2 = annotator_to_spans[annotator2]
            if not spans1 or not spans2:
                continue

            max_pos = max(s['end'] for spans in (spans1, spans2) for s in spans)
            array1 = np.zeros(max_pos + 1, dtype=bool)
            array2 = np.zeros(max_pos + 1, dtype=bool)
            for s in spans1:
                array1[s['start']:s['end']] = True
            for s in spans2:
                array2[s['start']:s['end']] = True

            tp = np.sum(array1 & array2)
            precision = tp / np.sum(array1) if np.sum(array1) > 0 else 0
            recall    = tp / np.sum(array2) if np.sum(array2) > 0 else 0
            f1 = (2 * precision * recall / (precision + recall)
                  if (precision + recall) > 0 else 0)

            doc_results['annotator_pairs'].append({
                'annotator1': annotator1, 'annotator2': annotator2,
                'precision': precision, 'recall': recall, 'f1': f1,
            })
            doc_p.append(precision); doc_r.append(recall); doc_f.append(f1)
            overall_p.append(precision); overall_r.append(recall); overall_f.append(f1)

        if doc_f:
            doc_results['avg_precision'] = float(np.mean(doc_p))
            doc_results['avg_recall']    = float(np.mean(doc_r))
            doc_results['avg_f1']        = float(np.mean(doc_f))
        results['per_document'][doc_id] = doc_results

    if overall_f:
        results['overall']['avg_precision']     = float(np.mean(overall_p))
        results['overall']['avg_recall']        = float(np.mean(overall_r))
        results['overall']['avg_f1']            = float(np.mean(overall_f))
        results['overall']['total_comparisons'] = len(overall_f)

    return results

In [ ]:
def calculate_category_agreement(task_id_to_annotations, attribute='labels'):
    """
    Cohen's κ on overlapping spans (≥ 50% overlap of the smaller span),
    weighted by the number of overlapping spans per pair.
    attribute: 'labels', 'subcategory', or 'impact'.
    """
    results = {'per_document': {}, 'overall': {'avg_kappa': None, 'pair_count': 0}}
    all_kappas, all_weights = [], []

    for doc_id, annotations in task_id_to_annotations.items():
        annotator_to_spans = {}
        for annotation in annotations:
            annotator = annotation.get('annotator_id')
            if not annotator:
                continue
            spans = []
            for label in annotation.get('enriched_labels', []):
                span = {'start': label.get('start'), 'end': label.get('end')}
                if attribute == 'labels':
                    span['category'] = (label.get('labels', [''])[0]
                                        if label.get('labels') else '')
                elif attribute == 'subcategory':
                    span['category'] = label.get('subcategory', '')
                elif attribute == 'impact':
                    span['category'] = label.get('impact', '')
                else:
                    continue
                spans.append(span)
            annotator_to_spans[annotator] = spans

        if len(annotator_to_spans) < 2:
            continue

        doc_kappas, doc_weights = [], []

        for annotator1, annotator2 in combinations(annotator_to_spans.keys(), 2):
            spans1 = annotator_to_spans[annotator1]
            spans2 = annotator_to_spans[annotator2]

            overlapping = []
            for s1 in spans1:
                for s2 in spans2:
                    if s1['start'] < s2['end'] and s2['start'] < s1['end']:
                        overlap_len = (min(s1['end'], s2['end']) -
                                       max(s1['start'], s2['start']))
                        min_len = min(s1['end'] - s1['start'],
                                      s2['end'] - s2['start'])
                        if min_len > 0 and overlap_len / min_len >= 0.5:
                            overlapping.append((s1['category'], s2['category']))

            if len(overlapping) < 2:
                continue

            cats1 = [o[0] for o in overlapping]
            cats2 = [o[1] for o in overlapping]
            try:
                kappa  = cohen_kappa_score(cats1, cats2)
                if np.isnan(kappa):
                    # All spans in one category for both annotators → κ undefined;
                    # treat as perfect agreement only if labels match exactly
                    kappa = 1.0 if cats1 == cats2 else 0.0
                weight = len(overlapping)
                doc_kappas.append(kappa); doc_weights.append(weight)
                all_kappas.append(kappa); all_weights.append(weight)
            except Exception as e:
                print(f"Warning: κ failed for {annotator1}/{annotator2} on {doc_id} — {e}")
                continue

        if doc_kappas:
            tw = sum(doc_weights)
            results['per_document'][doc_id] = {
                'avg_kappa': sum(k * w for k, w in zip(doc_kappas, doc_weights)) / tw,
                'pair_count': len(doc_kappas),
            }

    if all_kappas:
        tw = sum(all_weights)
        results['overall']['avg_kappa']  = (
            sum(k * w for k, w in zip(all_kappas, all_weights)) / tw)
        results['overall']['pair_count'] = len(all_kappas)

    return results

In [ ]:
def calculate_lead_time(task_id_to_annotations):
    """Mean / median lead time in minutes, overall and per annotator."""
    overall, per_annotator = [], {}
    for annotations in task_id_to_annotations.values():
        for a in annotations:
            lt = a.get('lead_time')
            if lt is None:
                continue
            minutes = lt / 60
            overall.append(minutes)
            per_annotator.setdefault(a['annotator_id'], []).append(minutes)
    return {
        'mean':   float(np.mean(overall))   if overall else None,
        'median': float(np.median(overall)) if overall else None,
        'per_annotator': {k: float(np.mean(v)) for k, v in per_annotator.items()},
    }


def calculate_annotation_density(task_id_to_annotations):
    """Mean spans marked per annotation, overall and per annotator."""
    overall, per_annotator = [], {}
    for annotations in task_id_to_annotations.values():
        for a in annotations:
            n = len(a.get('enriched_labels', []))
            overall.append(n)
            per_annotator.setdefault(a['annotator_id'], []).append(n)
    return {
        'mean': float(np.mean(overall)) if overall else None,
        'per_annotator': {k: float(np.mean(v)) for k, v in per_annotator.items()},
    }


def calculate_rating_agreement(task_id_to_annotations):
    """
    Mean absolute difference between annotator pairs on document-level
    correspondence and readability ratings (1–4 scale). 0 = perfect agreement.
    """
    corr_diffs, read_diffs = [], []
    for annotations in task_id_to_annotations.values():
        by_annotator = {a['annotator_id']: a for a in annotations}
        for a1, a2 in combinations(by_annotator.keys(), 2):
            r1, r2 = by_annotator[a1], by_annotator[a2]
            if (r1.get('overall_correspondence') is not None and
                    r2.get('overall_correspondence') is not None):
                corr_diffs.append(abs(r1['overall_correspondence'] -
                                      r2['overall_correspondence']))
            if (r1.get('overall_readability') is not None and
                    r2.get('overall_readability') is not None):
                read_diffs.append(abs(r1['overall_readability'] -
                                      r2['overall_readability']))
    return {
        'correspondence_mad': float(np.mean(corr_diffs)) if corr_diffs else None,
        'readability_mad':    float(np.mean(read_diffs)) if read_diffs else None,
    }

In [ ]:
# Compute all metrics for every project round
rows = []
for proj in PROJECTS:
    data = proj["data"]

    proj["exact"]   = calculate_exact_matches(data)
    proj["f1"]      = calculate_span_f1(data)
    proj["kappa"]   = calculate_category_agreement(data, attribute="labels")
    proj["timing"]  = calculate_lead_time(data)
    proj["density"] = calculate_annotation_density(data)
    proj["ratings"] = calculate_rating_agreement(data)

    rows.append({
        "project":            proj["label"],
        "date":               proj["date"],
        "n_files":            len(proj["files"]),
        "n_documents":        len(data),
        "n_annotations":      sum(len(v) for v in data.values()),
        "exact_match_rate":   proj["exact"]["overall"]["exact_match_rate"],
        "span_f1":            proj["f1"]["overall"]["avg_f1"],
        "span_precision":     proj["f1"]["overall"]["avg_precision"],
        "span_recall":        proj["f1"]["overall"]["avg_recall"],
        "cohens_kappa":       proj["kappa"]["overall"]["avg_kappa"],
        "lead_time_mean_min": proj["timing"]["mean"],
        "spans_per_annotation": proj["density"]["mean"],
        "correspondence_mad": proj["ratings"]["correspondence_mad"],
        "readability_mad":    proj["ratings"]["readability_mad"],
    })

trend_df = pd.DataFrame(rows)
display(trend_df.round(3))

## 5. Visualizations

The main dashboard shows the four headline metrics as line charts across project
rounds. Dashed vertical lines mark calibration meetings, so you can read each
segment as one full cycle: *evaluate → measure → calibrate → re-evaluate*.

Supporting charts add the per-annotator and contextual views:
lead time per annotator, annotation density, and document-level rating agreement.

In [ ]:
def fig_to_html(fig):
    """Encode a matplotlib figure to an inline base64 PNG img tag."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode("utf-8")
    plt.close(fig)
    return f'<img src="data:image/png;base64,{encoded}" style="max-width:100%;" />'


def _x_axis(ax, df):
    """Shared x-axis: project label with date underneath."""
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels([f"{r.project}\n{r.date}" for r in df.itertuples()],
                       fontsize=8.5)


def _calibration_markers(ax, df, label_first=False):
    """Dashed vertical line after each project followed by a calibration meeting."""
    drawn = False
    for i, r in enumerate(df.itertuples()):
        if r.project in CALIBRATION_AFTER and i < len(df) - 1:
            ax.axvline(x=i + 0.5, color="#888", linestyle="--",
                       linewidth=1, alpha=0.7,
                       label="Calibration meeting"
                       if (label_first and not drawn) else None)
            drawn = True


def chart_main_dashboard(df):
    """2×2 line-chart dashboard: exact match, span F1, Cohen's κ, lead time."""
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))

    panels = [
        (axes[0, 0], "exact_match_rate", "Exact Span Match Rate",
         "Proportion of unique spans", TARGETS.get("exact_match"), "#4C72B0", True),
        (axes[0, 1], "span_f1", "Span Agreement (Character-level F1)",
         "Mean pairwise F1", TARGETS.get("span_f1"), "#DD8452", True),
        (axes[1, 0], "cohens_kappa", "Cohen's κ — Error Category Agreement",
         "Weighted mean κ (overlapping spans)", TARGETS.get("kappa"), "#55A868", False),
        (axes[1, 1], "lead_time_mean_min", "Lead Time",
         "Mean minutes per annotation", TARGETS.get("lead_time"), "#8172B3", False),
    ]

    for ax, col, title, ylabel, target, color, pct in panels:
        y = df[col]
        ax.plot(range(len(df)), y, marker="o", color=color, linewidth=2,
                markersize=7, zorder=3)
        for i, v in enumerate(y):
            if pd.notna(v):
                txt = f"{v:.0%}" if pct else f"{v:.2f}"
                ax.annotate(txt, (i, v), textcoords="offset points",
                            xytext=(0, 9), ha="center", fontsize=8.5)
        if target is not None:
            ax.axhline(y=target, color="crimson", linestyle=":", alpha=0.7,
                       linewidth=1.5)
            trans = matplotlib.transforms.blended_transform_factory(
                ax.transAxes, ax.transData)
            ax.text(0.99, target,
                    f"target {target:.0%} " if pct else f"target {target:.2f} ",
                    transform=trans, color="crimson", fontsize=8,
                    va="bottom", ha="right")
        _calibration_markers(ax, df)
        _x_axis(ax, df)
        ax.set_title(title, fontsize=11.5)
        ax.set_ylabel(ylabel, fontsize=9)
        if pct:
            ax.yaxis.set_major_formatter(
                matplotlib.ticker.PercentFormatter(xmax=1.0, decimals=0))
        if col == "cohens_kappa":
            ax.axhline(y=0, color="gray", linewidth=0.8, alpha=0.6)

    # Headroom for the value labels
    for ax in axes.flat:
        lo, hi = ax.get_ylim()
        ax.set_ylim(lo, hi + (hi - lo) * 0.12)

    fig.suptitle(f"{PROJECT_NAME} — Agreement Trends Across Evaluation Rounds",
                 fontsize=14, y=0.995)
    fig.text(0.5, 0.955, "Dashed vertical lines = calibration meetings",
             ha="center", fontsize=9, color="#666", style="italic")
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    return fig


def _per_annotator_lines(metric_key, value_key, title, ylabel):
    """
    Generic per-annotator line chart across rounds.
    metric_key: 'timing' or 'density'; value_key: 'per_annotator'.
    """
    all_annotators = sorted({a for p in PROJECTS
                             for a in p[metric_key][value_key].keys()})
    fig, ax = plt.subplots(figsize=(11, 4.5))
    palette = sns.color_palette("deep", max(len(all_annotators), 3))

    for color, annotator in zip(palette, all_annotators):
        ys = [p[metric_key][value_key].get(annotator, np.nan) for p in PROJECTS]
        ax.plot(range(len(PROJECTS)), ys, marker="o", linewidth=1.8,
                color=color, label=f"Annotator {annotator}")

    _calibration_markers(ax, trend_df)
    _x_axis(ax, trend_df)
    ax.set_title(title, fontsize=11.5)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.legend(fontsize=8.5, ncol=min(len(all_annotators), 4))
    fig.tight_layout()
    return fig


def chart_lead_time_per_annotator():
    return _per_annotator_lines(
        "timing", "per_annotator",
        "Lead Time per Annotator", "Mean minutes per annotation")


def chart_density_per_annotator():
    return _per_annotator_lines(
        "density", "per_annotator",
        "Annotation Density per Annotator", "Mean spans per annotation")


def chart_rating_agreement(df):
    """Document-level rating agreement (lower = better)."""
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(range(len(df)), df["correspondence_mad"], marker="o",
            color="#4C72B0", linewidth=2, label="Correspondence")
    ax.plot(range(len(df)), df["readability_mad"], marker="s",
            color="#DD8452", linewidth=2, label="Readability")
    _calibration_markers(ax, df, label_first=True)
    _x_axis(ax, df)
    ax.set_title("Document-Level Rating Agreement "
                 "(mean |difference| between annotators — lower is better)",
                 fontsize=11.5)
    ax.set_ylabel("Mean absolute difference (1–4 scale)", fontsize=9)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=8.5)
    fig.tight_layout()
    return fig

In [ ]:
# Preview all charts inline
fig_dashboard = chart_main_dashboard(trend_df)
display(fig_dashboard)

fig_lead = chart_lead_time_per_annotator()
display(fig_lead)

fig_density = chart_density_per_annotator()
display(fig_density)

fig_ratings = chart_rating_agreement(trend_df)
display(fig_ratings)

## 6. Export Report

Assembles a self-contained HTML report (same styling as the other notebooks in
the series) and writes it to `OUTPUT_DIR`. The executive summary table includes
round-over-round deltas; ▲/▼ indicate the direction of change, with green/red
showing whether the change is an improvement for that metric (lower lead time
and lower rating MAD count as improvements).

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

safe_name = PROJECT_NAME.lower().replace(" ", "_")
base_path = os.path.join(OUTPUT_DIR,
                         f"{safe_name}_progress_{REPORT_DATE}.html")

report_path = base_path
counter = 1
while os.path.exists(report_path):
    root, ext = os.path.splitext(base_path)
    report_path = f"{root} ({counter}){ext}"
    counter += 1

# ── Executive summary table with deltas ──────────────────────────────────────
SUMMARY_METRICS = [
    ("exact_match_rate",   "Exact Match",          "pct",  "up"),
    ("span_f1",            "Span F1",              "num",  "up"),
    ("cohens_kappa",       "Cohen's κ",            "num",  "up"),
    ("lead_time_mean_min", "Lead Time (min)",      "num",  "down"),
    ("correspondence_mad", "Corr. Rating MAD",     "num",  "down"),
    ("readability_mad",    "Read. Rating MAD",     "num",  "down"),
]

def fmt(v, kind):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "—"
    return f"{v:.0%}" if kind == "pct" else f"{v:.2f}"

def delta_cell(curr, prev, kind, good_direction):
    if prev is None or curr is None or pd.isna(prev) or pd.isna(curr):
        return ""
    d = curr - prev
    if abs(d) < 1e-9:
        return ' <span style="color:#999;">●</span>'
    arrow    = "▲" if d > 0 else "▼"
    improved = (d > 0) if good_direction == "up" else (d < 0)
    color    = "#2e7d32" if improved else "#c62828"
    dtxt     = f"{abs(d):.0%}" if kind == "pct" else f"{abs(d):.2f}"
    return f' <span style="color:{color};font-size:.85em;">{arrow}{dtxt}</span>'

summary_rows = []
for i, r in enumerate(trend_df.itertuples()):
    cells = [f"<td><b>{r.project}</b><br/><span style='color:#777;font-size:.85em;'>{r.date}</span></td>",
             f"<td>{r.n_documents} docs / {r.n_annotations} ann.</td>"]
    prev = trend_df.iloc[i - 1] if i > 0 else None
    for col, _, kind, direction in SUMMARY_METRICS:
        curr_v = getattr(r, col)
        prev_v = prev[col] if prev is not None else None
        cells.append(f"<td>{fmt(curr_v, kind)}"
                     f"{delta_cell(curr_v, prev_v, kind, direction)}</td>")
    calib = "✔" if r.project in CALIBRATION_AFTER else ""
    cells.append(f"<td style='text-align:center;'>{calib}</td>")
    summary_rows.append("<tr>" + "".join(cells) + "</tr>")

summary_headers = (["Round", "Volume"] +
                   [m[1] for m in SUMMARY_METRICS] +
                   ["Calibration after"])
summary_table = (
    '<div class="table-wrap"><table><thead><tr>'
    + "".join(f"<th>{h}</th>" for h in summary_headers)
    + "</tr></thead><tbody>" + "".join(summary_rows) + "</tbody></table></div>"
)

# Auto-generated first→last narrative
first, last = trend_df.iloc[0], trend_df.iloc[-1]
narrative_bits = []
for col, name, kind, direction in SUMMARY_METRICS[:4]:
    f_v, l_v = first[col], last[col]
    if pd.isna(f_v) or pd.isna(l_v):
        continue
    narrative_bits.append(f"{name}: {fmt(f_v, kind)} → {fmt(l_v, kind)}")
exec_text = (f"From {first['project']} ({first['date']}) to "
             f"{last['project']} ({last['date']}) — " +
             "; ".join(narrative_bits) + ".")

# ── Per-annotator lead time table ────────────────────────────────────────────
all_annotators = sorted({a for p in PROJECTS
                         for a in p["timing"]["per_annotator"].keys()})
ann_rows = []
for p in PROJECTS:
    tds = [f"<td><b>{p['label']}</b></td>"]
    for a in all_annotators:
        v = p["timing"]["per_annotator"].get(a)
        tds.append(f"<td>{f'{v:.1f}' if v is not None else '—'}</td>")
    ann_rows.append("<tr>" + "".join(tds) + "</tr>")
ann_table = (
    '<div class="table-wrap"><table><thead><tr><th>Round</th>'
    + "".join(f"<th>Annotator {a} (min)</th>" for a in all_annotators)
    + "</tr></thead><tbody>" + "".join(ann_rows) + "</tbody></table></div>"
)

# ── Assemble HTML ─────────────────────────────────────────────────────────────
html_doc = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>{PROJECT_NAME} – Annotator Performance Over Time – {REPORT_DATE}</title>
  <style>
    body        {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
                   max-width: 1100px; margin: 2rem auto; padding: 0 1.5rem;
                   color: #222; line-height: 1.55; }}
    h1          {{ border-bottom: 3px solid #444; padding-bottom: .5rem; }}
    h2          {{ color: #333; margin-top: .5rem; margin-bottom: .5rem; }}
    h3          {{ color: #555; }}
    .meta       {{ color: #666; font-size: .9rem; margin-bottom: 2rem; }}
    .section-desc {{ color: #555; font-style: italic; font-size: .92rem;
                     margin: .4rem 0 1rem 0; }}
    section     {{ background: #fafafa; border: 1px solid #e0e0e0;
                   border-radius: 6px; padding: 1.2rem 1.5rem;
                   margin-bottom: 2rem; }}
    .table-wrap {{ overflow-x: auto; margin-top: .5rem; }}
    table       {{ border-collapse: collapse; white-space: nowrap; width: 100%; }}
    th, td      {{ border: 1px solid #ddd; padding: .35rem .7rem; text-align: left; }}
    thead tr    {{ background: #f0f0f0; }}
  </style>
</head>
<body>
  <h1>{PROJECT_NAME} – Annotator Performance Over Time</h1>
  <p class="meta">
    Report date: {REPORT_DATE} &nbsp;|&nbsp;
    Rounds: {len(PROJECTS)} ({trend_df.iloc[0]['project']} → {trend_df.iloc[-1]['project']}) &nbsp;|&nbsp;
    Total annotations: {int(trend_df['n_annotations'].sum())}
  </p>

  <section>
    <h2>1. Executive Summary</h2>
    <p class="section-desc">{exec_text}</p>
    <p class="section-desc">▲/▼ show change vs. the previous round
       (green = improvement, red = regression). ✔ marks rounds followed by a
       calibration meeting.</p>
    {summary_table}
  </section>

  <section>
    <h2>2. Agreement Trends</h2>
    <p class="section-desc">Dashed vertical lines mark calibration meetings.
       Each segment between lines is one full evaluation cycle.</p>
    {fig_to_html(chart_main_dashboard(trend_df))}
  </section>

  <section>
    <h2>3. Per-Annotator Views</h2>
    <p class="section-desc">Lead time and annotation density per annotator.
       Diverging density is worth raising in calibration — agreement metrics can
       move simply because one annotator marks far more or fewer spans.</p>
    {fig_to_html(chart_lead_time_per_annotator())}
    {ann_table}
    <br/>
    {fig_to_html(chart_density_per_annotator())}
  </section>

  <section>
    <h2>4. Document-Level Rating Agreement</h2>
    <p class="section-desc">Mean absolute difference between annotators'
       holistic correspondence and readability ratings (1–4 scale). Lower is
       better; 0 means identical ratings on every document.</p>
    {fig_to_html(chart_rating_agreement(trend_df))}
  </section>

  <section>
    <h2>5. Methodology Notes</h2>
    <p class="section-desc">
      Exact match, span F1, and Cohen's κ use the same calculations as the
      single-project agreement notebook (§5.1–5.3), pooled across all
      annotator pairs and documents in each round. Rounds with two translations
      pool both files; task IDs are namespaced per file so annotations of
      different translations are never compared against each other.
      κ is computed on spans overlapping ≥ 50% and weighted by pair size.
      Lead time is Label Studio's recorded annotation lead time.
    </p>
  </section>
</body>
</html>"""

with open(report_path, "w", encoding="utf-8") as f:
    f.write(html_doc)

print(f"Report written to: {report_path}")
display(HTML(f'<b>Report saved:</b> <code>{report_path}</code>'))